**Preparing the mock data**

In [18]:
import time
from dataclasses import dataclass,field
import torch
import torch.nn as nn

In [61]:
!wget -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-08-27 18:19:07--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-08-27 18:19:07 (32.8 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
torch.manual_seed(0)

In [4]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
len(text)

1115394

In [14]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")

In [15]:
tokens = enc.encode(text)
len(tokens)

338025

In [22]:
# this same list, callers just pass their own list through to override it.
DEFAULT_BUCKET = [64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768]
device = "cuda" if torch.cuda.is_available() else "cpu"
STACKED_STRATEGIES = {"dense_masked", "sort_and_pad", "sort_pad_bucket"}
vocab_size = 50,257

**Simple data_loader**

In [17]:
class DataLoaderLite:
    def __init__(self, B, T, process_rank, num_processes):
        self.B = B
        self.T = T
        self.process_rank = process_rank
        self.num_processes = num_processes

        with open("input.txt", "r") as f:
            text = f.read()

        enc = tiktoken.get_encoding("gpt2")
        self.tokens = enc.encode(text)

        self.current_position = B * T * self.process_rank

    def next_batch(self):
        B, T = self.B, self.T

        buf = torch.tensor(
            self.tokens[
                self.current_position:
                self.current_position + B * T + 1
            ]
        )

        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)

        self.current_position += B * T * self.num_processes

        if (
            self.current_position
            + B * T * self.num_processes
            + 1
            > len(self.tokens)
        ):
            self.current_position = B * T * self.process_rank

        return x, y

EXPERTS, ROUTINGS ,AND DISPATCH( THESE ARE THE MAIN ENGINE)

In [16]:
"""
Expert weight storage and computation.

Two representations, for different dispatch memory layouts:

  ModuleListExperts: a plain nn.ModuleList of independent FFN modules.
    Needed by strategies that index into an actual list of sub-modules
    (sort_and_slice, dense_masked).

  StackedExperts: all expert weights as single 3D parameter tensors
    (num_experts, ...), like Mixtral. Needed by grouped-GEMM-style
    dispatch, which batch-matmuls across the expert dimension instead
    of looping python-side over nn.Linear calls.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    """Single FFN expert: up-projection -> GELU -> down-projection.

      swiglu=True: f_u packs gate and up projections together
        (2 * hidden_dim), one matmul produces both halves via
        .chunk(2, dim=-1). forward computes GELU(gate) * up @ down.

    """

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, swiglu: bool = True) -> None:
        super().__init__()
        self.swiglu = swiglu
        up_dim = 2 * hidden_dim if swiglu else hidden_dim
        self.gelu = nn.GELU(approximate="tanh")
        self.f_u = nn.Linear(input_dim, up_dim, bias=False)
        self.f_d = nn.Linear(hidden_dim, output_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.swiglu:
            gate, up = self.f_u(x).chunk(2, dim=-1)
            hidden = self.gelu(gate) * up
        else:
            hidden = self.gelu(self.f_u(x))
        return self.f_d(hidden)

class ModuleListExperts(nn.Module):
    """A list of independent Expert modules, indexed by expert id."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        num_experts: int,
        swiglu: bool = True,
    ) -> None:
        super().__init__()
        self.experts = nn.ModuleList(
            [Expert(input_dim, hidden_dim, output_dim, swiglu=swiglu) for _ in range(num_experts)]
        )
        self.num_experts = num_experts

    def __getitem__(self, idx: int) -> Expert:
        return self.experts[idx]

    def __len__(self) -> int:
        return self.num_experts





class StackedExperts(nn.Module):
    """All expert weights as single 3D tensors: (num_experts, ...).
    Uses SwiGLU (GLU Variants Improve Transformer: https://arxiv.org/pdf/2002.05202).

      swiglu=True: gate_up_proj packs gate and up projections together
        (2 * hidden_dim), one matmul per expert produces both halves via
        .chunk(2, dim=-1). forward_one_expert computes GELU(gate) * up @ down.

      swiglu=False: plain FFN, GELU(up(x)) @ down, same function as
        experts.Expert, just stored as stacked tensors. Exists so
        dense_masked can be checked for numerical equivalence against
        dense_all_experts and sort_and_slice (both plain-FFN Expert).
        With swiglu=True the two architectures compute different
        functions and are not expected to match.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        num_experts: int,
        swiglu: bool = True,
    ) -> None:
        super().__init__()
        self.num_experts = num_experts
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.swiglu = swiglu

        up_width = 2 * hidden_dim if swiglu else hidden_dim
        self.gate_up_proj = nn.Parameter(torch.empty(num_experts, up_width, input_dim))
        self.down_proj = nn.Parameter(torch.empty(num_experts, output_dim, hidden_dim))
        self.gelu = nn.GELU(approximate="tanh")

        self._init_weights()

    def __len__(self) -> int:
        return self.num_experts

    def _init_weights(self) -> None:
        for e in range(self.num_experts):
            nn.init.normal_(self.gate_up_proj[e], mean=0.0, std=0.02)
            nn.init.normal_(self.down_proj[e], mean=0.0, std=0.02)

    def forward_one_expert(self, x: torch.Tensor, expert_idx: int) -> torch.Tensor:
        """Apply a single expert (by id) to a batch of tokens already
        gathered for that expert. Used by dispatch strategies that loop
        python-side over which experts were actually hit.
        """
        if self.swiglu:
            gate, up = F.linear(x, self.gate_up_proj[expert_idx]).chunk(2, dim=-1)
            hidden = self.gelu(gate) * up
        else:
            hidden = self.gelu(F.linear(x, self.gate_up_proj[expert_idx]))
        return F.linear(hidden, self.down_proj[expert_idx])
    def forward_batched(self, x: torch.Tensor) -> torch.Tensor:
        """Apply all experts at once via a single batched matmul.

        x: (num_experts, capacity, input_dim), each slice already routed
        to its corresponding expert (e.g. by sort_and_pad). Used by dispatch
        strategies that batch across the expert dimension instead of
        looping python-side -- see forward_one_expert for the per-expert
        equivalent.
        """
        if self.swiglu:
            gate, up = (x@self.gate_up_proj.transpose(1, 2)).chunk(2, dim=-1)
            hidden = self.gelu(gate) * up
        else:
            hidden = self.gelu(torch.bmm(x, self.gate_up_proj.transpose(1, 2)))
        return torch.bmm(hidden, self.down_proj.transpose(1, 2))


In [19]:
class GatingNetwork(nn.Module):
    def __init__(self, input_dim, num_experts):
        super().__init__()
        self.gate = nn.Linear(input_dim, num_experts)

    def forward(self, x):
        return self.gate(x)
@dataclass
class RoutingResult:
    """Output of a full routing pass for one layer, one forward call.

    logits:  (T, num_experts) raw gate logits, graph-attached
    probs:   (T, num_experts) softmax over all experts, graph-attached
    values:  (T, top_k) softmax probability of each chosen expert, graph-attached
    indices: (T, top_k) chosen expert ids, no grad (indices are not differentiable)
    """

    logits: torch.Tensor
    probs: torch.Tensor
    values: torch.Tensor
    indices: torch.Tensor


def route(
    x: torch.Tensor,
    gate: GatingNetwork,
    top_k: int,
) -> RoutingResult:
    """Run the gate and select top_k experts per token.

    x is expected already flattened to (T, C) flattening B,T into one
    token axis is dispatch's concern (different dispatch strategies want
    different shapes), not the router's.
    """
    logits = gate(x)
    probs = torch.softmax(logits, dim=-1)
    values, indices = torch.topk(probs, top_k, dim=-1)
    return RoutingResult(logits=logits, probs=probs, values=values, indices=indices)

In [20]:
"""
dispatch.py — token routing strategies for MoE layers.

Four strategies: dense_all_experts (reference/oracle), dense_masked
(Mixtral-style), sort_and_slice (fastest, used with global-LBL), and
sort_and_pad (capacity-bounded batched matmul, may drop tokens).

All four share the signature (x, routing, experts, top_k) -> (T, C),
except sort_and_pad which requires an extra `capacity` argument.
See DISPATCH_REGISTRY at the bottom to swap strategies by name.
"""

import torch



# Default capacity buckets for sort_pad_bucket. Kept here (not imported from
# moe.py's MOEConfig) to avoid a circular import: moe.py imports this module,
# so this module cannot import back from moe.py. MOEConfig.bucket defaults to
# this same list, callers just pass their own list through to override it.
DEFAULT_BUCKET = [64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768]


def dense_all_experts(
    x: torch.Tensor,
    routing: RoutingResult,
    experts: ModuleListExperts,
    top_k: int,
) -> torch.Tensor:
    """Naive reference dispatch: every expert sees every token.

    O(num_experts) full-width matmuls regardless of how sparse the actual
    routing is. Never use this for anything beyond a correctness check or
    a small toy demo -- it defeats the entire point of MoE sparsity.
    """
    T, C = x.shape     # T -> total number of tokens and C--> embedding/hidden dim

    num_experts = len(experts)

    expert_out_list = [expert(x).unsqueeze(1) for expert in experts.experts]  # each (T, 1, C)
    expert_output = torch.cat(expert_out_list, dim=1)  # (T, num_experts, C)

    weight_full = torch.zeros(T, num_experts, device=x.device, dtype=x.dtype)
    # placing the routing values at the position specified by the index for expert
    weight_full.scatter_(1, routing.indices, routing.values)  # (T, num_experts)


    out = torch.einsum("te,tec->tc", weight_full, expert_output)
    return out


def dense_masked(
    x: torch.Tensor,
    routing: RoutingResult,
    experts: StackedExperts,
    top_k: int,
) -> torch.Tensor:
    """dispatch: one-hot mask, loop only over hit experts.

    Builds a (num_experts, top_k, T) mask, finds which experts were
    selected by at least one token, and for each such expert gathers its
    tokens, computes, and scatters the weighted result back via
    index_add_. Skips experts with zero tokens but still does an explicit
    gather/scatter per hit expert.
    """
    T, C = x.shape
    num_experts = experts.num_experts
    final_output = torch.zeros(T, C, device=x.device, dtype=x.dtype)

    with torch.no_grad():
        expert_mask = torch.nn.functional.one_hot(routing.indices, num_classes=num_experts)
        expert_mask = expert_mask.permute(2, 1, 0)  # (num_experts, top_k, T)
        expert_used = expert_mask.sum(dim=(-1, -2)).nonzero()
    type(Expert)
    for expert_idx_tensor in expert_used:
        expert_idx = expert_idx_tensor[0].item()

        top_k_pos, token_idx = torch.where(expert_mask[expert_idx])
        current_state = x[token_idx]
        current_hidden = experts.forward_one_expert(current_state, expert_idx)
        current_hidden = current_hidden * routing.values[token_idx, top_k_pos, None]
        final_output.index_add_(0, token_idx, current_hidden.to(final_output.dtype))

    return final_output


def sort_and_slice(
    x: torch.Tensor,
    routing: RoutingResult,
    experts: ModuleListExperts,
    top_k: int,
) -> torch.Tensor:
    """Aimed to be Fastest dispatch: global sort by expert id, contiguous per-expert slices.

    Every (token, chosen-expert) pair is flattened into one list, sorted by
    expert id so all of expert i's tokens land in one contiguous block, and
    bincount gives the exact slice boundaries with no further masking. Each
    expert's forward call operates on a tightly packed (count_i, C) tensor
     no wasted computation on tokens the expert doesn't own, no gather
    via boolean indexing (argsort + contiguous slicing is cheaper than
    torch.where-based gathering).

    This is the dispatch paired with the global-LBL MoE variant, since that
    variant is meant to be the fastest end-to-end configuration.
    """
    T, C = x.shape
    num_experts = len(experts)

    flattened_indices = torch.flatten(routing.indices)
    flattened_values = torch.flatten(routing.values)
    token_ids = torch.arange(T, device=x.device).unsqueeze(1).expand(-1, top_k).reshape(-1)

    exp_order = torch.argsort(flattened_indices)
    sorted_experts = flattened_indices[exp_order]
    sorted_token_ids = token_ids[exp_order]
    sorted_values = flattened_values[exp_order]

    counts = torch.bincount(sorted_experts, minlength=num_experts)

    final_output = torch.zeros(T, C, device=x.device, dtype=x.dtype)
    start = 0
    for expert_id in range(num_experts):
        count = counts[expert_id].item()
        if count == 0:
            continue
        end = start + count
        tok_ids = sorted_token_ids[start:end]
        weights = sorted_values[start:end]
        expert_out = experts[expert_id](x[tok_ids])
        final_output[tok_ids] += expert_out * weights.unsqueeze(-1)
        start = end

    return final_output

def sort_and_pad(
    x: torch.Tensor,
    routing: RoutingResult,
    experts: StackedExperts,
    top_k: int,
    capacity_factor: float = 2, # fix: instead passing capcity pass capacity factor
) -> torch.Tensor:
    """Sort + fixed capacity buffer, single batched matmul across all experts.

    Same sort step as sort_and_slice, but each expert's slice is padded or
    truncated to `capacity` tokens, enabling one bmm over the full
    (num_experts, capacity, C) buffer instead of a Python loop.
    Tokens beyond capacity for their expert are silently dropped.
    Only numerically equivalent to the other strategies when capacity is large
    enough that no token is dropped for the given input.

    capacity = round((T * top_k / num_experts) * capacity_factor), same
    idea as Switch Transformer: one shared value for every expert, not
    measured from this batch's actual load. Tokens past capacity for
    their expert get dropped. Want dropless? see sort_pad_bucket
    (bucketed capacity) or sort_and_slice (no padding at all).
    """
    T, C = x.shape
    N = T * top_k
    num_experts = len(experts)
    assert capacity_factor > 0, "capacity_factor must be positive" # changed from capcity to capcity_factor
    assert routing.indices.shape == (T, top_k), routing.indices.shape

    capacity = round((N / num_experts) * capacity_factor) # calculate capacity


    flattened_indices = torch.flatten(routing.indices)
    flattened_values = torch.flatten(routing.values)
    token_ids = torch.arange(T, device=x.device).unsqueeze(1).expand(-1, top_k).reshape(-1)

    exp_order = torch.argsort(flattened_indices)
    sorted_experts = flattened_indices[exp_order]
    sorted_token_ids = token_ids[exp_order]
    sorted_values = flattened_values[exp_order]

    group_sizes = torch.bincount(sorted_experts, minlength=num_experts)
    group_starts = torch.cumsum(group_sizes, dim=0) - group_sizes
    local_rank = torch.arange(N, device=x.device) - torch.repeat_interleave(group_starts, group_sizes)

    keep_mask = local_rank < capacity
    kept_dest = sorted_experts[keep_mask] * capacity + local_rank[keep_mask]
    kept_token_ids = sorted_token_ids[keep_mask]

    padded = torch.zeros(num_experts * capacity, C, device=x.device, dtype=x.dtype)
    padded[kept_dest] = x[kept_token_ids]

    hidden = experts.forward_batched(padded.view(num_experts, capacity, C))
    out_flat = hidden.reshape(num_experts * capacity, -1)

    weighted = out_flat[kept_dest] * sorted_values[keep_mask].unsqueeze(-1)

    final_output = torch.zeros(T, C, device=x.device, dtype=x.dtype)
    final_output.scatter_add_(
        0, kept_token_ids.unsqueeze(-1).expand(-1, C), weighted.to(final_output.dtype)
    )
    return final_output

def sort_pad_bucket(
    x: torch.Tensor,
    routing: RoutingResult,
    experts: StackedExperts,
    top_k: int,
    BUCKET: list = None,
) -> torch.Tensor:
    """This func works mostly similar but with a slight optimisation, aimed to be dropless.

    Optimisation: sort_and_pad above is not aware of what capacity its
    receiving. It may be greater then the max group_size, which leads
    to a wasteful amount of padding, or it can be smaller then some of
    the group sizes, which leads to dropping tokens, which is not a
    desired thing. So instead of passing a hard capacity choice, we let
    the batch choose (we could have done max(group_size) every time, but
    this would have a lot of different shapes every time which are bad
    for hardware efficiency, so instead there is a fixed number of sizes
    we can pick from, this saves the caches and pytorch need not to
    recompile every time).

    MoEConfig defines a default BUCKET list, and it is optional, one can
    pass his own. If the batch's max group_size ends up bigger then every
    bucket in the list, this will raise, so the largest bucket should
    always cover the biggest batch this is ever run on.
    """

    if BUCKET is None:
        BUCKET = DEFAULT_BUCKET

    T, C = x.shape
    N = T * top_k
    num_experts = len(experts)
    assert len(BUCKET) > 0, "BUCKET must be non-empty"
    assert routing.indices.shape == (T, top_k), routing.indices.shape

    flattened_indices = torch.flatten(routing.indices)
    flattened_values = torch.flatten(routing.values)
    token_ids = torch.arange(T, device=x.device).unsqueeze(1).expand(-1, top_k).reshape(-1)

    exp_order = torch.argsort(flattened_indices)
    sorted_experts = flattened_indices[exp_order]
    sorted_token_ids = token_ids[exp_order]
    sorted_values = flattened_values[exp_order]

    group_sizes = torch.bincount(sorted_experts, minlength=num_experts)
    max_group_size = torch.max(group_sizes).item()
    # NOTE The largest value of capcity must be >= T(Total_tokens) as if the worst case happens and we have the max(bucket) < the worst casr
    # the next would stop iteration
    capacity = next(b for b in BUCKET if max_group_size <= b)

    group_starts = torch.cumsum(group_sizes, dim=0) - group_sizes

    local_rank = torch.arange(N, device=x.device) - torch.repeat_interleave(group_starts, group_sizes)

    keep_mask = local_rank < capacity
    kept_dest = sorted_experts[keep_mask] * capacity + local_rank[keep_mask]
    kept_token_ids = sorted_token_ids[keep_mask]

    padded = torch.zeros(num_experts * capacity, C, device=x.device, dtype=x.dtype)
    padded[kept_dest] = x[kept_token_ids]

    hidden = experts.forward_batched(padded.view(num_experts, capacity, C))
    out_flat = hidden.reshape(num_experts * capacity, -1)

    weighted = out_flat[kept_dest] * sorted_values[keep_mask].unsqueeze(-1)

    final_output = torch.zeros(T, C, device=x.device, dtype=x.dtype)
    final_output.scatter_add_(
        0, kept_token_ids.unsqueeze(-1).expand(-1, C), weighted.to(final_output.dtype)
    )
    return final_output


DISPATCH_REGISTRY = {
    "dense_all_experts": dense_all_experts,
    "dense_masked": dense_masked,
    "sort_and_slice": sort_and_slice,
    "sort_and_pad":      sort_and_pad,
    "sort_pad_bucket": sort_pad_bucket,
}


**BENCH_PREPARING**

In [21]:
def build_matched_experts(n_embd, hidden, num_experts, device):
    ml = ModuleListExperts(n_embd, hidden, n_embd, num_experts, swiglu=False).to(device)
    st = StackedExperts(n_embd, hidden, n_embd, num_experts, swiglu=False).to(device)
    with torch.no_grad():
        for e in range(num_experts):
            expert = ml[e]
            st.gate_up_proj[e] = expert.f_u.weight.detach().clone()
            st.down_proj[e] = expert.f_d.weight.detach().clone()
    return ml, st

In [25]:
def load_real_batch(B, T, n_embd,dataloader,device):
    """Real tinyshakespeare token ids, embedded to (B*T, n_embd) hidden
    states via a throwaway (untrained) nn.Embedding -- dispatch only
    cares about hidden-state shape/values, not that the embedding is
    trained, this just gives more realistic routing statistics than
    pure gaussian noise.
    """


    x_ids, _ = dataloader.next_batch()
    embed = torch.nn.Embedding(vocab_size(), n_embd).to(device)
    x = embed(x_ids).reshape(B * T, n_embd)
    x = x.detach().requires_grad_(True)
    return x

**EDIT: Bench2 actually not one as first u should check the equivalence and then time**

In [ ]:
  #BENCH1 : measuring the time for all the dispatch technique, majorly for cuda. This below is doing a forward and back_pass both so measuring time on both

In [24]:
def time_dispatch(fn, x, gate, experts, top_k, iters, device, extra_kwargs):
    """Each iteration re-runs routing fresh, since dispatch consumes (and
    frees) the routing tensors' graph on backward, reusing one
    RoutingResult object across iterations double-backwards through an
    already-freed graph. This also more realistically mirrors real
    training, where routing is recomputed every forward pass anyway, not
    cached across steps.
    """
    def run_once():
        routing = route(x, gate, top_k)
        out = fn(x, routing, experts, top_k, **extra_kwargs)
        out.sum().backward()
        x.grad = None
        for p in experts.parameters():
            p.grad = None
        for p in gate.parameters():
            p.grad = None
    #warmup
    for _ in range(5):
        run_once()

    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        run_once()
    if device.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    return (t1 - t0) / iters * 1000  # ms/iter

In [26]:
@dataclass
class MOEConfig:
    n_embd: int
    n_experts: int
    top_k: int
    dispatch: str = "sort_pad_bucket"
    alpha_moe: float = 0.01
    hidden_dim: int  = 4 * 384  # defaults to 4 * n_embd, see __post_init__
    capacity_factor: float = 2 # used by sort_and_pad
    bucket: list = field(default_factory=lambda: list(DEFAULT_BUCKET))  # used by sort_pad_bucket
    swiglu: bool = True


In [32]:
def load_real_batch(B, T_seq, n_embd,dataloader,device:str = "cuda"):
    """Real tinyshakespeare token ids, embedded to (B*T, n_embd) hidden
    states via a throwaway (untrained) nn.Embedding -- dispatch only
    cares about hidden-state shape/values, not that the embedding is
    trained, this just gives more realistic routing statistics than
    pure gaussian noise.
    """


    x_ids, _ = dataloader.next_batch()
    x_ids = x_ids.to(device)
    embed = torch.nn.Embedding(50257, n_embd).to(device)
    x = embed(x_ids).reshape(B * T_seq, n_embd)
    x = x.detach().requires_grad_(True)
    return x



In [33]:
def main(
    T: int = 16384,
    n_embd: int = 384,
    hidden: int = 4 * 384,
    num_experts: int = 8,
    top_k: int = 2,
    iters: int = 20,
    synthetic: bool = False,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(0)
    B = 8
    # make object once
    dataloader = DataLoaderLite(B,T // B,0,1)

    if synthetic:
        x = torch.randn(T, n_embd, device=device, requires_grad=True)
    else:
        B = 8
        T_seq = max(T // B, 1)
        x = load_real_batch(B, T_seq, n_embd,dataloader, "cuda")
        T = x.shape[0]  # B * T_seq, may differ slightly from requested T due to integer division
    print('worked till here')
    gate = GatingNetwork(n_embd, num_experts).to(device)
    ml, st = build_matched_experts(n_embd, hidden, num_experts, device)

    results = {}
    for name, fn in DISPATCH_REGISTRY.items():
        experts = st if name in STACKED_STRATEGIES else ml
        extra_kwargs = {}
        if name == "sort_pad_bucket":
            extra_kwargs = {"BUCKET": MOEConfig(n_embd=n_embd, n_experts=num_experts, top_k=top_k).bucket}
        ms = time_dispatch(fn, x, gate, experts, top_k, iters, device, extra_kwargs)
        results[name] = ms

    print(
        f"device={device}, T={T}, n_embd={n_embd}, num_experts={num_experts}, top_k={top_k}, "
        f"iters={iters}, data={'synthetic' if synthetic else 'shakespeare'}\n"
    )
    for name, ms in sorted(results.items(), key=lambda kv: kv[1]):
        print(f"  {name:20s} {ms:8.3f} ms/iter")
    gate_logits = gate(x)

    return results,gate_logits

In [34]:
results,gate_logits  = main()


worked till here
device=cuda, T=16384, n_embd=384, num_experts=8, top_k=2, iters=20, data=shakespeare

  dense_masked           70.204 ms/iter
  sort_and_slice         71.609 ms/iter
  sort_and_pad          114.330 ms/iter
  sort_pad_bucket       114.892 ms/iter
  dense_all_experts     220.336 ms/iter


In [ ]:
#below i did some debugging so thats it

In [35]:
gate_logits.shape

torch.Size([16384, 8])

In [36]:
routings  = torch.softmax(gate_logits,dim=-1)

In [37]:
values , indices = torch.topk(routings,k=2,dim=-1)

In [38]:
indices.shape

torch.Size([16384, 2])

In [39]:
num_experts = 8

In [40]:
flattened_indices = torch.flatten(indices)
counts = torch.bincount(flattened_indices, minlength=num_experts)

In [41]:
counts

tensor([5669, 3937, 6341, 3184, 3406, 3626, 3482, 3123], device='cuda:0')

In [42]:
with torch.no_grad():
        expert_mask = torch.nn.functional.one_hot(indices, num_classes=num_experts)
        expert_mask = expert_mask.permute(2, 1, 0)  # (num_experts, top_k, T)
        expert_used = expert_mask.sum(dim=(-1, -2)).nonzero()

In [43]:
expert_used

tensor([[0],
        [1],
        [2],
        [3],
        [4],
        [5],
        [6],
        [7]], device='cuda:0')



```
# this is torch.bmm version
```  dense_masked           94.179 ms/iter
  sort_and_slice         96.740 ms/iter
  sort_and_pad          105.847 ms/iter
  sort_pad_bucket       157.410 ms/iter
  dense_all_experts     291.403 ms/iter


this is without torch.bmm ( may be seed changed )
worked till here
device=cuda, T=16384, n_embd=384, num_experts=8, top_k=2, iters=20, data=shakespeare

  dense_masked           91.398 ms/iter
  sort_and_slice         93.611 ms/iter
  sort_and_pad          101.518 ms/iter
  sort_pad_bucket       153.695 ms/iter
  dense_all_experts     283.791 ms/iter
# same capcity for pad and bucket
dense_masked           79.634 ms/iter
  sort_and_slice         81.992 ms/iter
  sort_and_pad          128.139 ms/iter
  sort_pad_bucket       129.997 ms/iter
  dense_all_experts     247.470 ms/iter


**THIS should run first before time_bench**


In [ ]:
# this is simply checking if all the dispatch techniques are are equivalent in giving the final result
# in this version the sort and pad version and sort_pad_bucket version explicitly forced to have a same capacity although one they sort_and_pad can
# differ in the ral training with dynamic gate_logits and dropping of tokens

In [57]:
def _extra_kwargs(name: str, T: int) -> dict:
    if name == "sort_and_pad":
        return {"capacity_factor": 2.0}  # deliberately oversized to guarantee no drops
    if name == "sort_pad_bucket":
        return {"BUCKET": [T]}  # single bucket exactly at T covers the true worst case, dropless
    return {}

In [58]:
@torch.no_grad()
def check_all(T=4096, n_embd=384, hidden=4*384, num_experts=8, top_k=2, tol=1e-4, verbose=True):
    """Run every DISPATCH_REGISTRY strategy against the dense_all_experts
    oracle on one shared (x, routing) pair, return {name: max_abs_diff}.
    Raises AssertionError on the first strategy that disagrees past tol.
    """
    torch.manual_seed(0)
    device = torch.device("cpu")
    torch.manual_seed(0)
    x = torch.randn(T, n_embd, device=device)
    gate = GatingNetwork(n_embd, num_experts).to(device)
    routing = route(x, gate, top_k)

    ml, st = build_matched_experts(n_embd, hidden, num_experts, device)

    out_oracle = dense_all_experts(x, routing, ml, top_k)

    diffs = {}
    for name, fn in DISPATCH_REGISTRY.items():
        if name == "dense_all_experts":
            continue
        experts = st if name in STACKED_STRATEGIES else ml
        kwargs = _extra_kwargs(name, T)
        out = fn(x, routing, experts, top_k, **kwargs)
        diff = (out_oracle - out).abs().max().item()
        diffs[name] = diff
        if verbose:
            print(f"  max |dense_all_experts - {name:16s}| = {diff:.2e}")
        assert diff < tol, f"{name} disagrees with reference dispatch (diff={diff:.2e}, tol={tol:.2e})"

    if verbose:
        print(f"OK: all {len(diffs)} dispatch strategies agree with the oracle.")
    return diffs

In [59]:
diffs = check_all()

  max |dense_all_experts - dense_masked    | = 2.98e-08
  max |dense_all_experts - sort_and_slice  | = 2.98e-08
  max |dense_all_experts - sort_and_pad    | = 2.98e-08
  max |dense_all_experts - sort_pad_bucket | = 2.98e-08
OK: all 4 dispatch strategies agree with the oracle.
